In [ ]:
!pip install mlflow==3.14.0 dagshub==0.7.0 skops==0.14.0 pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 polars


# **1. Perkenalan Dataset**
**Dataset URL**: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Description**: This dataset contains historical customer data from a fictional telecommunications company in California. It includes 7,043 rows and 21 features capturing customer demographics, signed services (internet, phone, security), account settings (contract type, payment method), and charges, with the target column 'Churn' indicating whether the customer left the service within the quarter.

# **2. Import Library**
Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


# **3. Memuat Dataset**
Pada tahap ini, Anda perlu memuat dataset ke dalam notebook.

In [ ]:
data_path = 'raw_data.csv'
df = pd.read_csv(data_path)
df.head()


# **4. Exploratory Data Analysis (EDA)**
Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

In [ ]:
print("--- Data Info ---")
df.info()

print("\n--- Data Description ---")
display(df.describe(include='all'))

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Churn Distribution ---")
print(df['Churn'].value_counts())

sns.countplot(data=df, x='Churn')
plt.title('Distribution of Target Column (Churn)')
plt.show()


# **5. Data Preprocessing**
Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

In [ ]:
# Drop the customerID column
df = df.drop(columns=['customerID'])

# Crucial Cleaning: Convert TotalCharges to numeric, coerce errors, and fill NaNs with median
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
median_total_charges = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_total_charges)

# Convert target column Churn to binary integers (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Cleanly encode categorical features
df = pd.get_dummies(df, drop_first=True)

# Perform an 80:20 Train-Test split
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

try:
    out_dir = 'D:/dicoding/SMSML_Lukman/Membangun_model/namadataset_preprocessing'
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'cleaned_data.csv')
    cleaned_df = pd.concat([X, y], axis=1)
    cleaned_df.to_csv(out_path, index=False)
    print(f"Fully processed numeric-only data saved to: {out_path}")
except Exception as e:
    print(f"Could not save to local Windows path. Falling back to Colab /content/. Error: {e}")
    out_path = '/content/cleaned_data.csv'
    cleaned_df = pd.concat([X, y], axis=1)
    cleaned_df.to_csv(out_path, index=False)
    print(f"Fully processed numeric-only data saved to: {out_path}")


# **6. Model Training & Tracking**
Pada tahap ini kita melatih model dengan MLflow dan RandomizedSearchCV.

In [ ]:
# Retrieve DAGSHUB_CLIENT_TOKEN or fallback to MLFLOW_TRACKING_PASSWORD
dagshub_token = os.getenv("DAGSHUB_CLIENT_TOKEN") or os.getenv("MLFLOW_TRACKING_PASSWORD")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = "lukmannurh"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# Set the MLflow tracking URI headlessly
mlflow.set_tracking_uri("https://dagshub.com/lukmannurh/Eksperimen_SML_Lukman.mlflow")

# Activate autologging
mlflow.sklearn.autolog(log_models=True)

print("Starting MLflow experiment tracking session...")
with mlflow.start_run(run_name="Telco_Churn_Optimization"):
    
    # Initialize model
    rf = RandomForestClassifier(random_state=42)
    
    # Implement a controlled hyperparameter sweep
    param_dist = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }
    
    search = RandomizedSearchCV(
        estimator=rf, 
        param_distributions=param_dist, 
        n_iter=5, 
        cv=3, 
        scoring='accuracy',
        random_state=42,
        n_jobs=-1
    )
    
    # Train model
    search.fit(X_train, y_train)
    
    # Extract champion model
    champion_model = search.best_estimator_
    print("\nBest parameters found:", search.best_params_)
    
    # Predict on test set
    y_pred = champion_model.predict(X_test)
    
    # Calculate evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\nAccuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # Register the optimized champion model into the Model Registry
    mlflow.sklearn.log_model(
        sk_model=champion_model,
        artifact_path="champion_model_artifacts",
        registered_model_name="credit-scoring-model"
    )
    print("\nOptimized champion model successfully registered under 'credit-scoring-model'.")


# **7. Evaluation Metrics**


In [ ]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn (0)', 'Churn (1)'], 
            yticklabels=['No Churn (0)', 'Churn (1)'])
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()
